In [1]:
import duckdb
from networks.feed_forward_more_features_larger import NeuralModel
from networks.cnn_network import build_dataloaders

con = duckdb.connect('../capillary.db')

df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


print(f"Antal fall med M-komponent:{(df['label'] == 1).sum()}")
print(f"Antal fall utan M-komponent:{(df['label'] == 0).sum()}")


train_rows = df[df['set'] == 'train']
val_rows = df[df['set'] == 'val']
test_rows = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

network = NeuralModel()
network.reset_weights()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
network.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal fall med M-komponent:2942
Antal fall utan M-komponent:69882
Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 469,782
Epoch   0 | train: 0.6737 | val: 0.4567 | acc: 96.38% | AUC: 0.738  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_larger.pth
Epoch   1 | train: 0.5893 | val: 0.5722 | acc: 95.92% | AUC: 0.812  | LR: 0.001
Epoch   2 | train: 0.5965 | val: 0.6098 | acc: 67.17% | AUC: 0.831  | LR: 0.001
Epoch   3 | train: 0.5122 | val: 0.3665 | acc: 97.11% | AUC: 0.825  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_larger.pth
Epoch   4 | train: 0.5285 | val: 0.6110 | acc: 76.24% | AUC: 0.665  | LR: 0.001
Epoch   5 | train: 0.5338 | val: 0.4343 | acc: 94.37% | AUC: 0.858  | LR: 0.001
Epoch   6 | train: 0.4888 | val: 0.5248 | acc: 93.73% | AUC: 0.842  | LR: 0.001
Epoch   7 | train: 0.4914 | val: 0.5813 | acc: 74.81% | AUC: 0.921  | LR: 0.001
Epoch   8 | trai

In [2]:
con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id,fractions, boundaries, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


network = NeuralModel()
network.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 469,782
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
Epoch   0 | train: 0.6909 | val: 0.5573 | acc: 79.69% | AUC: 0.765  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_larger_fold1.pth
Epoch   1 | train: 0.6350 | val: 0.6083 | acc: 88.10% | AUC: 0.706  | LR: 0.001
Epoch   2 | train: 0.6029 | val: 0.6488 | acc: 89.41% | AUC: 0.727  | LR: 0.001
Epoch   3 | train: 0.5950 | val: 0.5508 | acc: 89.41% | AUC: 0.778  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_more_features_larger_fold1.pth
Epoch   4 | train: 0.5325 | val: 0.6436 | acc: 91.36% | AUC: 0.812  | LR: 0.001
Epoch   5 | train: 0.6133 | val: 0.6370 | acc: 87.64% | AUC: 0.736  | LR: 0.001
Epoch   6 | train: 0.6254 | val: 0.6543 | acc: 88.94% | AUC: 0.736  | LR: 0.001
Epoch   7 | train: 0.5766 | val: 0.5050 | acc: 85.83% | AUC: 0.823  | LR: 0.001
  -

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.166474,0.209523,92.565056,0.968505,0.932554,0.877323,1756,127,33,236
1,2,0.140776,0.239570,93.450999,0.956793,0.944268,0.866171,1779,105,36,233
2,3,0.188802,0.221617,93.082637,0.960461,0.936340,0.892193,1765,120,29,240
3,4,0.177697,0.217528,94.289694,0.965312,0.956499,0.847584,1803,82,41,228
4,5,0.186497,0.236634,94.289694,0.963547,0.955438,0.855019,1801,84,39,230
5,6,0.216403,0.283217,94.568245,0.959776,0.963926,0.817844,1817,68,49,220
6,7,0.187110,0.288244,90.427509,0.939320,0.911843,0.851301,1717,166,40,229
7,8,0.185924,0.212566,93.361188,0.965939,0.940053,0.888476,1772,113,30,239
8,9,0.149909,0.192043,94.800371,0.970536,0.952760,0.914815,1795,89,23,247
9,10,0.159934,0.161976,95.496750,0.981816,0.960191,0.918519,1809,75,22,248


In [3]:
from functions.evaluation import evaluate


test_rows = test_rows[test_rows['label'].isin([0,1])]
test_rows['cnn_probability'] = test_rows['probability'].copy()
result = network.predict(test_rows)
_ = evaluate(result)

KeyError: 'probability'

In [6]:
import pandas as pd
df = pd.read_csv('../models/feed_forward_more_features_larger_kfold_metrics.csv')
print(df['val_accuracy'].std())

1.4298048781326085
